In [14]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [15]:
df = pd.read_csv('IDTA.csv')

In [16]:
cols = [
    "Gender",
    "Sleep Duration",
    "Dietary Habits",
    "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness",
    "Depression"
]

In [17]:
data = df[cols].copy()

In [36]:
data_encoded = pd.get_dummies(data.astype(str))

print("Encoded columns:")
print(data_encoded.columns.tolist())

Encoded columns:
['Gender_Female', 'Gender_Male', 'Sleep Duration_5-6 hours', 'Sleep Duration_7-8 hours', 'Sleep Duration_Less than 5 hours', 'Sleep Duration_More than 8 hours', 'Dietary Habits_Healthy', 'Dietary Habits_Moderate', 'Dietary Habits_Unhealthy', 'Have you ever had suicidal thoughts ?_No', 'Have you ever had suicidal thoughts ?_Yes', 'Family History of Mental Illness_No', 'Family History of Mental Illness_Yes', 'Depression_No', 'Depression_Yes']


In [ ]:
frequent_itemsets = apriori(
    data_encoded,
    min_support=0.03,      
    use_colnames=True
)

print("\nNumber of frequent itemsets:", len(frequent_itemsets))
print(frequent_itemsets.head())


Number of frequent itemsets: 575
    support                            itemsets
0  0.481013                     (Gender_Female)
1  0.518987                       (Gender_Male)
2  0.245862          (Sleep Duration_5-6 hours)
3  0.258033          (Sleep Duration_7-8 hours)
4  0.255599  (Sleep Duration_Less than 5 hours)


In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3       
)

print("\nTotal rules generated:", len(rules))
print(rules.head())


Total rules generated: 2017
                          antecedents               consequents  \
0          (Sleep Duration_5-6 hours)           (Gender_Female)   
1          (Sleep Duration_7-8 hours)           (Gender_Female)   
2  (Sleep Duration_Less than 5 hours)           (Gender_Female)   
3  (Sleep Duration_More than 8 hours)           (Gender_Female)   
4                     (Gender_Female)  (Dietary Habits_Healthy)   

   antecedent support  consequent support   support  confidence      lift  \
0            0.245862            0.481013  0.121227    0.493069  1.025065   
1            0.258033            0.481013  0.116845    0.452830  0.941410   
2            0.255599            0.481013  0.128530    0.502857  1.045414   
3            0.240506            0.481013  0.114411    0.475709  0.988973   
4            0.481013            0.331548  0.162123    0.337045  1.016578   

   representativity  leverage  conviction  zhangs_metric   jaccard  certainty  \
0               1.0  0.0

In [ ]:
dep_yes_col = [c for c in data_encoded.columns if "Depression_Yes" in c][0]
print("\nDepression YES column name:", dep_yes_col)

rules_dep_yes = rules[
    rules["consequents"].apply(lambda x: dep_yes_col in list(x))
]

print("Rules with Depression=Yes on RHS:", len(rules_dep_yes))


if len(rules_dep_yes) == 0:
    print("\n⚠ No rules found with Depression=Yes on RHS. Showing top 5 rules overall instead.\n")
    top5 = rules.sort_values(by="confidence", ascending=False).head(5)
else:
   
    top5 = rules_dep_yes.sort_values(by="confidence", ascending=False).head(5)

print("\nTop 5 rules (raw):")
print(top5[["antecedents", "consequents", "support", "confidence", "lift"]])


Depression YES column name: Depression_Yes
Rules with Depression=Yes on RHS: 0

⚠ No rules found with Depression=Yes on RHS. Showing top 5 rules overall instead.


Top 5 rules (raw):
                                            antecedents      consequents  \
1594  (Sleep Duration_More than 8 hours, Have you ev...  (Depression_No)   
1774  (Family History of Mental Illness_No, Gender_F...  (Depression_No)   
793   (Sleep Duration_7-8 hours, Gender_Female, Have...  (Depression_No)   
1666  (Family History of Mental Illness_No, Have you...  (Depression_No)   
1908  (Gender_Male, Family History of Mental Illness...  (Depression_No)   

       support  confidence     lift  
1594  0.050146         1.0  1.10967  
1774  0.046738         1.0  1.10967  
793   0.053554         1.0  1.10967  
1666  0.093476         1.0  1.10967  
1908  0.031646         1.0  1.10967  


In [43]:
print("\nTop 5 rules (readable):\n")

if len(top5) == 0:
    print("No rules to display. Try lowering min_support or min_threshold.")
else:
    for i, row in top5.iterrows():
        antecedent = ", ".join(list(row["antecedents"]))
        consequent = ", ".join(list(row["consequents"]))
        print(f"IF {antecedent}  THEN  {consequent}")
        print(f"  support    = {row['support']:.3f}")
        print(f"  confidence = {row['confidence']:.3f}")
        print(f"  lift       = {row['lift']:.3f}")
        print("-" * 129)



Top 5 rules (readable):

IF Sleep Duration_More than 8 hours, Have you ever had suicidal thoughts ?_No, Dietary Habits_Healthy  THEN  Depression_No
  support    = 0.050
  confidence = 1.000
  lift       = 1.110
---------------------------------------------------------------------------------------------------------------------------------
IF Family History of Mental Illness_No, Gender_Female, Have you ever had suicidal thoughts ?_No, Dietary Habits_Healthy  THEN  Depression_No
  support    = 0.047
  confidence = 1.000
  lift       = 1.110
---------------------------------------------------------------------------------------------------------------------------------
IF Sleep Duration_7-8 hours, Gender_Female, Have you ever had suicidal thoughts ?_No  THEN  Depression_No
  support    = 0.054
  confidence = 1.000
  lift       = 1.110
---------------------------------------------------------------------------------------------------------------------------------
IF Family History of Ment